In [1]:
try:
    print(f"Spark version {sc.version}")
except:
    %run ../spark-instance.ipynb

SparkConf created
Started SparkSession
Spark version 3.5.3
You should be able to access the Spark UI at: https://dacs-compute-gate.ewi.utwente.nl:9999/user/g.luvizottocesar@utwente.nl/proxy/4040/stages/
Note that you may have to Enable extensions first via the Extension Manager.


In [15]:
clean_spark()

CLEANING SPARK INSTANCE...


In [2]:
import logging
from datetime import datetime
import time

import ipaddress

import pyspark.sql.types as pst
import pyspark.sql.functions as psf
from pyspark.sql.window import Window

from census_helper import download_date

In [3]:
sc.setCheckpointDir("checkpoint/")

In [4]:
# Create logger
logger = logging.getLogger("anycast_services")
logger.setLevel(logging.INFO)

# Prevent duplicate logs if logger already has handlers
if not logger.handlers:
    # Create file handler
    file_handler = logging.FileHandler("logs.log", encoding='utf-8')
    file_handler.setLevel(logging.INFO)
    
    # Create formatter
    formatter = logging.Formatter(
        fmt='%(asctime)s %(levelname)-8s %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    
    # Add formatter to handler
    file_handler.setFormatter(formatter)
    
    # Add handler to logger
    logger.addHandler(file_handler)
else:
    print("logger already exist!")

# Optionally, prevent propagation to root logger to avoid duplicates
logger.propagate = False

### IPv6 hitlist from OpenINTEL

In [5]:
# Base prefix of fDNS warehouse data
FDNS_WAREHOUSE_BASE = "s3a://openintel/category=fdns/type=warehouse"
# Prefix format() template for (source, date)-partition fDNS warehouse data
FDNS_SOURCE_DT_PARTITION_FMT_TEMPLATE = os.path.join(FDNS_WAREHOUSE_BASE, "source={source}", "year={year}", "month={month:02d}", "day={day:02d}")

In [7]:
sources = [
    "tranco",
     "com", "net", "org", "info", "mobi", "gov", "name", "asia", "nl", "se", "nu", "ca", "fi", "at", "dk", "us", "gt", "na", "ee", "co", "ch", "li", "sk", "fr", "cl", "czds",  # exists
    "umbrella", "radar", "majestic", "crux", # exists
    "infra:ns", "infra:mx",  # exists
    "biz",
    "closedcc",
    "imp",
    "infra:tlsa",
    "ip6-chli",
    "opencc",
    "root",
    "su",

    #"ct-logs", "fed.us", "bis", "aero", # does not exist
    #"alexa",  # not available for the given date
    # "ru", "xn--p1ai",  # ignore

]

snapshot = datetime(2026, 2, 21)

In [9]:
for source in sources:
    t1 = time.time()

    try:
        df = spark.read.parquet(f"{FDNS_WAREHOUSE_BASE}/source={source}/year={snapshot.year}/month={snapshot.month:02d}/day={snapshot.day:02d}")
    except:
        print(f"{source} not available")
        continue

    df.filter(
        (psf.col("query_type") == "AAAA")
        & (psf.col("ip6_address").isNotNull())
    ).select("ip6_address").distinct(
    ).write.parquet(f"s3a://luvizottocesarg-tmp/anycast-service-discovery/oi/ipv6_addresses/source={source}/year={snapshot.year}/month={snapshot.month:02d}/day={snapshot.day:02d}/")

    t2 = time.time()

    print(f"{source} done in {round(t2-t1, 1)} seconds")
    logger.info(f"{source} done in {round(t2-t1, 1)} seconds")


tranco done in 14.7 seconds
com done in 121.3 seconds
net done in 20.6 seconds
org done in 21.7 seconds
info done in 8.8 seconds
mobi done in 3.3 seconds
gov done in 2.0 seconds
name done in 2.3 seconds
asia done in 4.3 seconds
nl done in 20.7 seconds
se done in 6.6 seconds
nu done in 3.4 seconds
ca done in 7.8 seconds
fi done in 4.7 seconds
at done in 5.3 seconds
dk done in 5.9 seconds
us done in 7.4 seconds
gt done in 1.8 seconds
na done in 2.0 seconds
ee done in 3.3 seconds
co done in 9.0 seconds
ch done in 8.7 seconds
li done in 2.2 seconds
sk done in 4.7 seconds
fr done in 13.5 seconds
cl done in 5.8 seconds
czds done in 38.3 seconds
umbrella done in 10.6 seconds
radar done in 9.7 seconds
majestic done in 7.4 seconds
crux done in 28.4 seconds
infra:ns done in 4.8 seconds
infra:mx done in 20.9 seconds
biz done in 5.4 seconds
closedcc done in 26.3 seconds
imp done in 10.8 seconds
infra:tlsa done in 1.2 seconds
ip6-chli done in 1.7 seconds
opencc done in 19.8 seconds
root done in 1.2

In [13]:
df = spark.read.option("basePath", "s3a://luvizottocesarg-tmp/anycast-service-discovery/oi/ipv6_addresses/").parquet("s3a://luvizottocesarg-tmp/anycast-service-discovery/oi/ipv6_addresses/"
).filter(
    (psf.col("year") == snapshot.year)
    & (psf.col("month") == snapshot.month)
    & (psf.col("day") == snapshot.day)
)

print(df.select("ip6_address").distinct().count())

18490656


In [14]:
df.select("ip6_address").distinct().coalesce(1
).write.csv(f"s3a://luvizottocesarg-tmp/anycast-service-discovery/oi/ipv6_addresses/source=all/year={snapshot.year}/month={snapshot.month:02d}/day={snapshot.day:02d}")